# WikiArt Inpainting – Colab Training Pipeline

Notebook odtwarza pipeline z repozytorium WikiArt_Inpainting: **ekstrakcja cech → klasteryzacja → inpainting**.


## 🔎 Kroki w skrócie
1. Instalacja i konfiguracja środowiska Colab.
2. Wczytanie danych WikiArt i przygotowanie splitów.
3. Trening autoenkodera i ekstrakcja embeddingów.
4. Klasteryzacja embeddingów (PCA + KMeans/GMM/DBSCAN).
5. Trening bazowego modelu inpainting.
6. (Opcjonalnie) fine-tuning osobnych modeli dla klastrów.


In [ ]:
# 🔧 Setup (Colab)
# Jeśli uruchamiasz notebook lokalnie, możesz pominąć klonowanie.
from pathlib import Path

repo_dir = Path('/content/unsupervised-learning')
if not repo_dir.exists():
    !git clone https://github.com/adamzsl/unsupervised-learning.git

%cd /content/unsupervised-learning
!pip install -r requirements.txt


In [ ]:
# 📦 Importy i konfiguracja
from pathlib import Path
import pickle

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.utils import load_config, set_seed
from src.dataset import (
    DatasetSplits,
    WikiArtMaskedDataset,
    create_dataloaders,
    load_wikiart_splits,
)
from src.encoder.autoencoder import ConvAutoencoder, AutoencoderConfig, extract_embeddings
from src.clustering.clusterizer import cluster_embeddings
from src.inpainting.model import SimpleUNet, InpaintConfig
from src.inpainting.training import inpaint_step

config = load_config('configs/config.yaml')
set_seed(config['training']['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Wczytanie danych WikiArt
Możesz trenować na pełnym zbiorze (Artificio/WikiArt_Full), ale w Colabie warto użyć mniejszego podzbioru na start.


In [ ]:
data_cfg = config['data']
mask_cfg = config['damage']

splits = load_wikiart_splits(
    data_cfg['dataset_name'],
    train_split=data_cfg['train_split'],
    val_split=data_cfg['val_split'],
    test_split=data_cfg['test_split'],
    seed=config['training']['seed'],
)

# ✅ Opcjonalnie ogranicz liczbę próbek, aby przyspieszyć trening w Colabie
MAX_TRAIN = 5000
MAX_VAL = 1000
MAX_TEST = 1000

splits = DatasetSplits(
    train=splits.train.select(range(min(MAX_TRAIN, len(splits.train)))),
    val=splits.val.select(range(min(MAX_VAL, len(splits.val)))),
    test=splits.test.select(range(min(MAX_TEST, len(splits.test)))),
)

train_loader, val_loader, test_loader = create_dataloaders(
    splits,
    image_size=data_cfg['image_size'],
    mask_type=mask_cfg['mask_type'],
    batch_size=data_cfg['batch_size'],
    num_workers=data_cfg['num_workers'],
    max_damage_ratio=mask_cfg['max_damage_ratio'],
    min_mask_size=mask_cfg['min_mask_size'],
    max_mask_size=mask_cfg['max_mask_size'],
    brush_width=mask_cfg['brush_width'],
    num_strokes=mask_cfg['num_strokes'],
)

print('Train/Val/Test:', len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))


## 2. Trening autoenkodera (ekstrakcja cech)
Autoenkoder służy do uzyskania reprezentacji obrazu, które później klasteryzujemy.


In [ ]:
auto_cfg = AutoencoderConfig(
    input_channels=3,
    latent_dim=config['encoder']['latent_dim'],
    image_size=data_cfg['image_size'],
)

autoencoder = ConvAutoencoder(auto_cfg).to(device)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=config['encoder']['learning_rate'])
criterion = nn.L1Loss()

EPOCHS = 5  # zwiększ, jeśli chcesz pełny trening

for epoch in range(1, EPOCHS + 1):
    autoencoder.train()
    train_loss = 0.0
    for images, _, _, _ in tqdm(train_loader, desc=f'Train {epoch}/{EPOCHS}'):
        images = images.to(device)
        optimizer.zero_grad(set_to_none=True)
        recon, _ = autoencoder(images)
        loss = criterion(recon, images)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)

    train_loss /= max(1, len(train_loader.dataset))
    print(f'Epoch {epoch}: train_loss={train_loss:.4f}')

Path('models').mkdir(exist_ok=True)
torch.save(autoencoder.state_dict(), 'models/autoencoder.pth')


## 3. Ekstrakcja embeddingów i klasteryzacja
Z embeddingów tworzymy klastry, które wykorzystamy do fine-tuningu impaintera.


In [ ]:
embedding_loader = DataLoader(
    WikiArtMaskedDataset(
        splits.train,
        image_size=data_cfg['image_size'],
        mask_type=mask_cfg['mask_type'],
        max_damage_ratio=mask_cfg['max_damage_ratio'],
        min_mask_size=mask_cfg['min_mask_size'],
        max_mask_size=mask_cfg['max_mask_size'],
        brush_width=mask_cfg['brush_width'],
        num_strokes=mask_cfg['num_strokes'],
    ),
    batch_size=data_cfg['batch_size'],
    shuffle=False,
    num_workers=data_cfg['num_workers'],
)

autoencoder.eval()
embeddings = extract_embeddings(autoencoder, embedding_loader, device).numpy()

labels, artifacts = cluster_embeddings(
    embeddings,
    method=config['clustering']['method'],
    n_clusters=config['clustering']['n_clusters'],
)

np.save('models/train_cluster_labels.npy', labels)
with open('models/cluster_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('Cluster labels:', np.unique(labels))


## 4. Trening bazowego modelu inpainting
Najpierw uczymy model na całym zbiorze, a potem możemy go dostroić per-klaster.


In [ ]:
inpaint_model = SimpleUNet(InpaintConfig(base_channels=config['inpainting']['base_channels'])).to(device)
optimizer = torch.optim.Adam(inpaint_model.parameters(), lr=1e-3)

INPAINT_EPOCHS = 5

for epoch in range(1, INPAINT_EPOCHS + 1):
    inpaint_model.train()
    epoch_loss = 0.0
    for batch in tqdm(train_loader, desc=f'Inpaint {epoch}/{INPAINT_EPOCHS}'):
        optimizer.zero_grad(set_to_none=True)
        loss_dict = inpaint_step(inpaint_model, batch, device)
        loss = loss_dict['loss']
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch[0].size(0)

    epoch_loss /= max(1, len(train_loader.dataset))
    print(f'Epoch {epoch}: loss={epoch_loss:.4f}')

torch.save(inpaint_model.state_dict(), 'models/inpaint_base.pth')


## 5. Fine-tuning per klaster (opcjonalne)
Każdy klaster trenujemy osobno. To odpowiada podejściu z WikiArt_Inpainting.


In [ ]:
from collections import defaultdict

cluster_indices = defaultdict(list)
for idx, label in enumerate(labels):
    cluster_indices[int(label)].append(idx)

CLUSTER_EPOCHS = 2  # zwiększ dla lepszych wyników
MIN_CLUSTER_SIZE = 200

for cluster_id, indices in cluster_indices.items():
    if len(indices) < MIN_CLUSTER_SIZE:
        print(f'Skip cluster {cluster_id} (size={len(indices)})')
        continue

    subset = splits.train.select(indices)
    cluster_ds = WikiArtMaskedDataset(
        subset,
        image_size=data_cfg['image_size'],
        mask_type=mask_cfg['mask_type'],
        max_damage_ratio=mask_cfg['max_damage_ratio'],
        min_mask_size=mask_cfg['min_mask_size'],
        max_mask_size=mask_cfg['max_mask_size'],
        brush_width=mask_cfg['brush_width'],
        num_strokes=mask_cfg['num_strokes'],
    )
    cluster_loader = DataLoader(
        cluster_ds,
        batch_size=data_cfg['batch_size'],
        shuffle=True,
        num_workers=data_cfg['num_workers'],
    )

    cluster_model = SimpleUNet(InpaintConfig(base_channels=config['inpainting']['base_channels'])).to(device)
    cluster_model.load_state_dict(inpaint_model.state_dict())

    optimizer = torch.optim.Adam(cluster_model.parameters(), lr=1e-4)

    for epoch in range(1, CLUSTER_EPOCHS + 1):
        cluster_model.train()
        epoch_loss = 0.0
        for batch in tqdm(cluster_loader, desc=f'Cluster {cluster_id} ({epoch}/{CLUSTER_EPOCHS})'):
            optimizer.zero_grad(set_to_none=True)
            loss_dict = inpaint_step(cluster_model, batch, device)
            loss = loss_dict['loss']
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch[0].size(0)

        epoch_loss /= max(1, len(cluster_loader.dataset))
        print(f'Cluster {cluster_id} Epoch {epoch}: loss={epoch_loss:.4f}')

    torch.save(cluster_model.state_dict(), f'models/inpaint_cluster_{cluster_id}.pth')


## 6. Szybki test impaintera
Użyj modelu bazowego lub jednego z modeli klastrowych.


In [ ]:
import matplotlib.pyplot as plt

sample_ds = WikiArtMaskedDataset(
    splits.test,
    image_size=data_cfg['image_size'],
    mask_type=mask_cfg['mask_type'],
    max_damage_ratio=mask_cfg['max_damage_ratio'],
    min_mask_size=mask_cfg['min_mask_size'],
    max_mask_size=mask_cfg['max_mask_size'],
    brush_width=mask_cfg['brush_width'],
    num_strokes=mask_cfg['num_strokes'],
)

image, mask, masked, _ = sample_ds[0]

with torch.no_grad():
    output = inpaint_model(masked.unsqueeze(0).to(device), mask.unsqueeze(0).to(device))

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image.permute(1, 2, 0))
axes[0].set_title('Oryginał')
axes[1].imshow(masked.permute(1, 2, 0))
axes[1].set_title('Uszkodzony')
axes[2].imshow(output.squeeze(0).cpu().permute(1, 2, 0))
axes[2].set_title('Rekonstrukcja')
for ax in axes:
    ax.axis('off')
plt.show()
